# House Price Prediction – Flask API Deployment

This notebook packages an **already-trained** house-price regression model behind a Flask REST API.

## Before you run
1. Put your trained model file (`.joblib` or `.pkl`) somewhere inside the notebook's working folder, or use the upload step below.
2. The model should ideally be the **complete preprocessing + regression pipeline** if training used scaling, encoding, or other transformations.
3. Run cells from top to bottom.
4. Use real feature names and valid sample values from your training dataset when testing `/predict`.

> This notebook cannot invent or reconstruct your original trained model or its input columns. If you have not trained/saved a model yet, train it in your original notebook first and save it with `joblib.dump(...)`.


## 1. Install dependencies

In [ ]:
%pip install flask joblib numpy pandas requests scikit-learn

## 2. Import libraries and set up paths

In [ ]:
import os
import sys
import time
import json
import joblib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests

PROJECT_DIR = Path.cwd() / "House_Price_Flask"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook working directory:", Path.cwd())
print("Project directory:", PROJECT_DIR.resolve())

## 3. Find or upload your trained model

Run the next cell to search the current folder and its subfolders. If it finds multiple model files, choose the correct one explicitly. If it finds none, upload/copy your trained `.joblib` or `.pkl` file into the notebook's working directory and rerun this cell.

If you have not saved your model yet, go to the training notebook and run (replace `model` with your actual trained model/pipeline variable):

```python
import joblib
joblib.dump(model, "house_price_model.joblib")
```


In [ ]:
# Search for saved models in the notebook folder.
model_files = sorted(
    p for p in Path.cwd().rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".joblib", ".pkl"}
    and PROJECT_DIR not in p.parents
)

print("Model files found:")
for i, path in enumerate(model_files):
    print(f"{i}: {path.resolve()}")

if not model_files:
    raise FileNotFoundError(
        "No .joblib or .pkl model found. Save/export your trained model, "
        "place it in the notebook folder (or a subfolder), then rerun this cell."
    )

if len(model_files) == 1:
    MODEL_SOURCE = model_files[0]
    print("Selected model:", MODEL_SOURCE)
else:
    print("Multiple models found. Set MODEL_SOURCE_NAME in the next cell.")


### 3A. Select the correct model (only needed if multiple were found)

In [ ]:
# If exactly one model was found, leave MODEL_SOURCE_NAME = None.
# If multiple were found, enter the exact relative or absolute path of the correct model.
MODEL_SOURCE_NAME = None  # Example: "models/house_price_model.joblib"

if MODEL_SOURCE_NAME is not None:
    MODEL_SOURCE = Path(MODEL_SOURCE_NAME)
    if not MODEL_SOURCE.is_absolute():
        MODEL_SOURCE = Path.cwd() / MODEL_SOURCE
    if not MODEL_SOURCE.is_file():
        raise FileNotFoundError(f"Selected model file does not exist: {MODEL_SOURCE}")
elif len(model_files) == 1:
    MODEL_SOURCE = model_files[0]
else:
    raise RuntimeError(
        "Choose the correct trained model by setting MODEL_SOURCE_NAME above."
    )

model = joblib.load(MODEL_SOURCE)
MODEL_PATH = PROJECT_DIR / "model.joblib"
joblib.dump(model, MODEL_PATH)

print("Loaded model:", MODEL_SOURCE.resolve())
print("Model class:", type(model).__name__)
print("Deployment copy:", MODEL_PATH.resolve())

## 4. Inspect the model's expected input features

In [ ]:
feature_names = getattr(model, "feature_names_in_", None)
number_of_features = getattr(model, "n_features_in_", None)

print("Expected number of features:", number_of_features)
if feature_names is not None:
    print("Expected feature names in training order:")
    for i, name in enumerate(feature_names, start=1):
        print(f"{i}. {name}")
else:
    print("This model does not expose feature_names_in_.")
    print("For prediction, supply a numeric list in the exact training-time order.")

## 5. Create the Flask application (`app.py`)

In [ ]:
app_code = r"""
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "model.joblib"

logging.basicConfig(level=logging.INFO)
app = Flask(__name__)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

model = joblib.load(MODEL_PATH)
FEATURE_NAMES = getattr(model, "feature_names_in_", None)
EXPECTED_FEATURES = getattr(model, "n_features_in_", None)


@app.get("/")
def home():
    return jsonify({
        "status": "success",
        "message": "House Price Prediction API is running.",
        "prediction_endpoint": "/predict"
    }), 200


@app.get("/health")
def health():
    return jsonify({
        "status": "healthy",
        "model_loaded": True
    }), 200


@app.post("/predict")
def predict():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Send a valid JSON object."}), 400

    if "features" not in data:
        return jsonify({"error": "Missing 'features' in JSON input."}), 400

    features = data["features"]

    try:
        if isinstance(features, dict):
            if FEATURE_NAMES is None:
                return jsonify({
                    "error": "Model has no stored feature names. Send an ordered numeric list."
                }), 400

            expected = list(FEATURE_NAMES)
            received = set(features.keys())
            missing = [name for name in expected if name not in received]
            extra = [name for name in received if name not in expected]

            if missing or extra:
                return jsonify({
                    "error": "Input features do not match the trained model.",
                    "missing_features": missing,
                    "unexpected_features": extra
                }), 400

            input_data = pd.DataFrame(
                [[features[name] for name in expected]],
                columns=expected
            )

        elif isinstance(features, list):
            if EXPECTED_FEATURES is not None and len(features) != int(EXPECTED_FEATURES):
                return jsonify({
                    "error": "Incorrect number of features.",
                    "expected": int(EXPECTED_FEATURES),
                    "received": len(features)
                }), 400

            if not all(
                isinstance(value, (int, float)) and not isinstance(value, bool)
                for value in features
            ):
                return jsonify({
                    "error": "List features must contain only numeric values."
                }), 400

            values = np.asarray(features, dtype=float)
            if values.ndim != 1 or not np.isfinite(values).all():
                return jsonify({
                    "error": "Features must be a one-dimensional list of finite numbers."
                }), 400

            if FEATURE_NAMES is not None:
                input_data = pd.DataFrame([values], columns=list(FEATURE_NAMES))
            else:
                input_data = values.reshape(1, -1)

        else:
            return jsonify({
                "error": "'features' must be a JSON object or numeric list."
            }), 400

        prediction = model.predict(input_data)
        predicted_price = float(np.asarray(prediction).reshape(-1)[0])

        if not np.isfinite(predicted_price):
            raise ValueError("Model returned a non-finite prediction.")

        return jsonify({
            "status": "success",
            "predicted_price": predicted_price
        }), 200

    except (ValueError, TypeError) as error:
        return jsonify({"error": str(error)}), 400
    except Exception:
        app.logger.exception("Prediction failed.")
        return jsonify({"error": "Internal prediction error."}), 500


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False)
"""

app_file = PROJECT_DIR / "app.py"
app_file.write_text(app_code, encoding="utf-8")
print("Created:", app_file.resolve())

## 6. Create `requirements.txt`

In [ ]:
requirements = """Flask>=3.0,<4.0
joblib>=1.3
numpy>=1.26
pandas>=2.0
scikit-learn>=1.3
requests>=2.31
"""

(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print("Created requirements.txt")

## 7. Create `README.md`

In [ ]:
readme = """# House Price Prediction API

## Overview
Deploys a trained house-price regression model as a Flask REST API.

## Technologies
Python, Flask, scikit-learn, Joblib, NumPy, Pandas, Requests.

## Install
    pip install -r requirements.txt

## Run
    python app.py

Base URL: http://127.0.0.1:5000

## Endpoints
- GET / : API welcome message
- GET /health : API health check
- POST /predict : generate a house-price prediction

## Prediction request
Send Content-Type: application/json.

If the model stores feature names:
    {
      "features": {
        "actual_feature_name_1": 100,
        "actual_feature_name_2": 3
      }
    }

If the model does not store feature names:
    {
      "features": [100, 3]
    }

Replace example names and values with the exact inputs used by the trained model.

## Response
    {
      "status": "success",
      "predicted_price": 250000.0
    }

The price above is illustrative only.
"""

(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Created README.md")

## 8. Start the Flask server from this notebook

In [ ]:
BASE_URL = "http://127.0.0.1:5000"
server_log_path = PROJECT_DIR / "server.log"
server_log = open(server_log_path, "w", encoding="utf-8")

server_process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=str(PROJECT_DIR),
    stdout=server_log,
    stderr=subprocess.STDOUT
)

server_ready = False
for _ in range(20):
    if server_process.poll() is not None:
        server_log.flush()
        raise RuntimeError(f"Flask stopped. Check {server_log_path}")
    try:
        health_response = requests.get(BASE_URL + "/health", timeout=2)
        if health_response.status_code == 200:
            server_ready = True
            break
    except requests.RequestException:
        time.sleep(0.5)

if not server_ready:
    raise RuntimeError(f"Flask did not start. Check {server_log_path}")

print("Flask is running at:", BASE_URL)
print("Health response:", health_response.json())

## 9. Prepare a prediction payload

The following creates a **template only**. Replace placeholder values with realistic values from your dataset. For categorical columns, provide categories in the exact form expected by the model/pipeline.


In [ ]:
if feature_names is not None:
    sample_features = {str(name): 1 for name in feature_names}
    print(json.dumps({"features": sample_features}, indent=4))
else:
    if number_of_features is None:
        raise RuntimeError("Model does not expose its expected feature count.")
    sample_features = [1.0] * int(number_of_features)
    print(json.dumps({"features": sample_features}, indent=4))

print("Replace placeholder values before treating the prediction as meaningful.")

## 10. Test the prediction endpoint

In [ ]:
payload = {"features": sample_features}

prediction_response = requests.post(
    BASE_URL + "/predict",
    json=payload,
    timeout=10
)

print("HTTP status:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=4))
prediction_response.raise_for_status()

## 11. Test invalid input

In [ ]:
invalid_response = requests.post(
    BASE_URL + "/predict",
    json={},
    timeout=10
)

print("HTTP status:", invalid_response.status_code)
print("Response:", invalid_response.json())

assert invalid_response.status_code == 400
print("Missing-features validation test passed.")

## 12. Test with curl (optional)

Keep Flask running, then open a terminal:

**Health check**
```bash
curl http://127.0.0.1:5000/health
```

**Prediction**
```bash
curl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" -d "{\"features\":{\"actual_feature_name_1\":100,\"actual_feature_name_2\":3}}"
```

Replace the sample feature names and values. For a model without stored feature names, use an ordered numeric list in the exact training-time order.


## 13. Verify project files and stop Flask

In [ ]:
required_files = ["app.py", "model.joblib", "requirements.txt", "README.md"]

for filename in required_files:
    file_path = PROJECT_DIR / filename
    print(f"{filename}: {'PASS' if file_path.is_file() else 'FAIL'}")

assert all((PROJECT_DIR / name).is_file() for name in required_files)
print("\nAll required project files are present.")

if server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=5)

server_log.close()
print("Flask server stopped.")

## Troubleshooting

- **No model found:** Save the trained model from your training notebook using `joblib.dump(model, "house_price_model.joblib")`, then put that file beside this notebook.
- **Multiple model files:** Set `MODEL_SOURCE_NAME` in section 3A to the exact correct file.
- **Feature mismatch:** Use the exact training feature names and order.
- **Categorical values fail:** Deploy the complete preprocessing pipeline, not just the final regression estimator.
- **Connection refused:** Make sure the Flask server cell completed successfully and is still running.
- **Port 5000 busy:** Stop the other server or change the port in both `app.py` and `BASE_URL`.

## Final submission checklist

- [ ] `app.py`
- [ ] `model.joblib` (your actual trained model/pipeline)
- [ ] `requirements.txt`
- [ ] `README.md`
- [ ] `House_Price_Flask_Project.ipynb` (this notebook, saved with executed outputs)

**Accuracy note:** The API code is a reusable deployment template. Exact successful predictions depend on using your actual trained model, its preprocessing, its feature schema, and valid sample data. A particular automated code-quality percentage cannot be guaranteed without knowing the evaluator's rubric.
